# Dating App Recommendation System

I want a simple, honest solution to one question:
which profiles should I show each user so that a match is more likely?

**Short version**

- Treat likes as implicit positive feedback on a user-to-user graph
  (decidermemberid liking othermemberid).
- Build a sparse user–item interaction matrix of likes.
- Learn latent factors with matrix factorisation (truncated SVD).
- For each user, rank candidate profiles by dot product in latent space,
  and hide profiles they already saw or liked in training.
- Evaluate with Hit Rate@K and MRR@K using a simple temporal hold-out.

The rest of the notebook just walks through that, step by step.


In [2]:
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import svds
import random

RANDOM_SEED = 42
REFERENCE_YEAR = 2021  # dataset window
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


## 1. Load and inspect the swipe data

The dataset covers five days of swipe behaviour from UK users.
Each row is a swipe from `decidermemberid` on `othermemberid`.

Fields I care about:

- `decidermemberid`: user doing the swipe.
- `othermemberid`: profile being swiped on.
- `timestamp`: UTC timestamp of the swipe.
- `like`: 1 = like (right swipe), 0 = pass.
- `decidergender` / `othergender`: gender for each side.
- `deciderdobyear` / `otherdobyear`: year of birth.
- `decidersignuptimestamp` / `othersignuptimestamp`: signup time.

For the recommender, I focus on likes and treat them as implicit positives.


In [3]:
import pathlib

# Use relative path - assumes notebook is in same directory as swipes.csv
DATA_PATH = pathlib.Path('swipes.csv')

swipes = pd.read_csv(
    DATA_PATH,
    parse_dates=[
        "timestamp",
        "decidersignuptimestamp",
        "othersignuptimestamp",
    ],
)

print("Raw shape:", swipes.shape)
swipes.head()


Raw shape: (9859578, 10)


,decidermemberid,othermemberid,timestamp,like,decidergender,othergender,deciderdobyear,otherdobyear,decidersignuptimestamp,othersignuptimestamp
0,3847776,3227524,2021-01-01 00:00:06,1,M,F,2002,1999,2020-12-28 11:19:02,2020-08-17 12:46:39
1,608590,519321,2021-01-01 00:00:06,0,F,M,1996,1992,2018-08-10 22:55:32,2018-06-17 15:34:37
2,397116,453914,2021-01-01 00:00:06,0,F,M,1991,1991,2018-03-04 22:14:06,2018-04-25 21:22:26
3,3847776,1269455,2021-01-01 00:00:06,1,M,F,2002,2000,2020-12-28 11:19:02,2019-04-22 20:02:54
4,1630969,347909,2021-01-01 00:00:23,0,F,M,1980,1983,2019-08-29 00:07:22,2018-01-17 12:14:03


Quick check on how many likes there are and how many unique users.


In [4]:
swipes["like"].value_counts(dropna=False), swipes["decidermemberid"].nunique(), swipes["othermemberid"].nunique()


(like
 0    6446515
 1    3413063
 Name: count, dtype: int64,
 52091,
 168862)

## 2. Filter to likes and create a time-aware train and validation split

I treat likes as implicit positive feedback and ignore passes for training.

To mimic a real swipe timeline:

- sort likes by timestamp,
- for each user with at least two likes, hold out the last like as validation,
- everything else stays in the training set,
- users with only one like are kept fully in training.

This keeps the split simple and respects time, which is enough for this exercise.


In [5]:
swipes_like = swipes[swipes["like"] == 1].copy()
print("Total likes:", len(swipes_like))


Total likes: 3413063


In [6]:
def train_val_split(df: pd.DataFrame, user_col: str = "decidermemberid") -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Temporal hold-out split:
    - For each user with at least two likes, hold out their last like (by timestamp).
    - All remaining likes go into train.
    - Users with fewer than two likes are fully in train.
    """
    df = df.sort_values("timestamp")

    user_counts = df[user_col].value_counts()
    multi_like_users = user_counts[user_counts >= 2].index

    is_multi = df[user_col].isin(multi_like_users)
    rank_desc = df[is_multi].groupby(user_col)["timestamp"].rank(
        method="first",
        ascending=False,
    )

    val_idx = df[is_multi].loc[rank_desc == 1].index

    val_df = df.loc[val_idx]
    train_df = df.drop(index=val_idx)

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


train_swipes, val_swipes = train_val_split(swipes_like)

print("Train likes:", len(train_swipes))
print("Validation likes:", len(val_swipes))


Train likes: 3372527
Validation likes: 40536


## 3. Build the interaction matrix and ID mappings

Next I turn the likes into a sparse user–item matrix:

- rows are `decidermemberid` (the user swiping),
- columns are `othermemberid` (the profile they saw),
- entries are 1 if there was a like.

This matrix drives the factorisation model.


In [7]:
def build_id_mappings(
    df: pd.DataFrame,
    user_col: str = "decidermemberid",
    item_col: str = "othermemberid",
):
    """
    Map raw member IDs to contiguous integer indices for users and items.
    Keeps matrices dense in index space and easier to work with.
    """
    user_ids = df[user_col].unique()
    item_ids = df[item_col].unique()

    user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
    item_to_idx = {iid: i for i, iid in enumerate(item_ids)}
    idx_to_user = {i: uid for uid, i in user_to_idx.items()}
    idx_to_item = {i: iid for iid, i in item_to_idx.items()}

    return user_to_idx, item_to_idx, idx_to_user, idx_to_item


def build_interaction_matrix(
    df: pd.DataFrame,
    user_to_idx: dict,
    item_to_idx: dict,
    user_col: str = "decidermemberid",
    item_col: str = "othermemberid",
) -> sparse.csr_matrix:
    """
    Build a sparse CSR user–item matrix from likes.
    Each (user, item) pair gets a value of 1.
    """
    user_idx = df[user_col].map(user_to_idx)
    item_idx = df[item_col].map(item_to_idx)

    mask = user_idx.notna() & item_idx.notna()
    user_idx = user_idx[mask].astype(int)
    item_idx = item_idx[mask].astype(int)

    data = np.ones(len(user_idx), dtype=np.float32)

    n_users = len(user_to_idx)
    n_items = len(item_to_idx)

    interaction = sparse.csr_matrix(
        (data, (user_idx, item_idx)),
        shape=(n_users, n_items),
    )
    return interaction


user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_mappings(
    train_swipes,
    user_col="decidermemberid",
    item_col="othermemberid",
)

interaction_matrix = build_interaction_matrix(
    train_swipes,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
    user_col="decidermemberid",
    item_col="othermemberid",
)

n_users, n_items = interaction_matrix.shape
print(f"Users: {n_users:,}, Items: {n_items:,}, Train interactions: {interaction_matrix.nnz:,}")


Users: 45,588, Items: 76,633, Train interactions: 3,219,332


## 4. Model: matrix factorisation with truncated SVD

At this point I treat the problem as implicit-feedback matrix factorisation.

Idea:

- compress the huge sparse matrix into low dimensional user and item vectors,
- dot products of those vectors score how compatible a user and a profile look,
- higher score, higher chance they should see each other.

I use truncated SVD for this:

- compute `R ≈ U S V^T`,
- define user factors as `U * sqrt(S)`,
- define item factors as `V^T * sqrt(S)`,
- score(user, item) is the dot product of those two vectors.


In [8]:
def train_svd_model(interaction: sparse.csr_matrix, n_factors: int = 32):
    """
    Learn low rank user and item factors from the interaction matrix using truncated SVD.
    """
    u, s, vt = svds(interaction, k=n_factors)
    order = np.argsort(-s)
    s = s[order]
    u = u[:, order]
    vt = vt[order, :]

    sqrt_s = np.sqrt(s)
    user_factors = u * sqrt_s
    item_factors = (vt.T) * sqrt_s

    return user_factors, item_factors


user_factors, item_factors = train_svd_model(interaction_matrix, n_factors=32)


## 5. Ranking and evaluation with Hit Rate@K and MRR@K

I care about ranking quality: for each user, how high do we place a held out
positive example in the recommendation list.

Metrics:

- **Hit Rate@K**  
  fraction of users whose held out like shows up in the top K recommendations.

- **MRR@K** (Mean Reciprocal Rank)  
  for each user, take 1 divided by the rank of the held out profile within the
  top K list (or 0 if it is not there), then average across users.

This lines up with the product behaviour: if you rarely surface relevant people
near the top, your swipe experience feels flat and slow.


In [9]:
def _topk_for_user(
    user_idx: int,
    k: int,
    interaction: sparse.csr_matrix,
    user_factors: np.ndarray,
    item_factors: np.ndarray,
    extra: int = 200,
) -> list[int]:
    """
    Get top k item indices for a given user index:
    - score items by dot product in latent space,
    - drop items that the user already liked in training,
    - 'extra' controls how many candidates to consider before filtering.
    """
    user_vec = user_factors[user_idx]
    if not np.any(user_vec):
        return []

    scores = item_factors @ user_vec

    seen = set(interaction[user_idx].indices)

    candidate_count = min(len(scores), k + extra)
    if candidate_count <= 0:
        return []

    top_idx = np.argpartition(-scores, candidate_count - 1)[:candidate_count]
    ordered = top_idx[np.argsort(-scores[top_idx])]

    filtered = [idx for idx in ordered if idx not in seen]
    return filtered[:k]


In [10]:
def evaluate_model(
    interaction: sparse.csr_matrix,
    val_df: pd.DataFrame,
    user_to_idx: dict,
    item_to_idx: dict,
    user_factors: np.ndarray,
    item_factors: np.ndarray,
    k: int = 10,
):
    """
    Offline evaluation over validation likes.
    For each user with a held out like:
    - generate top K recommendations,
    - check if the held out profile is present and at what rank.
    """
    if val_df.empty:
        return {
            "users_evaluated": 0,
            "users_skipped": 0,
            f"hit_rate@{k}": np.nan,
            f"mrr@{k}": np.nan,
        }

    hits = []
    reciprocal_ranks = []
    users_skipped = 0

    grouped = val_df.groupby("decidermemberid")

    for member_id, group in grouped:
        if member_id not in user_to_idx:
            continue

        user_idx = user_to_idx[member_id]

        if interaction[user_idx].nnz == 0:
            users_skipped += 1
            continue

        target_items = [
            item_to_idx[other_id]
            for other_id in group["othermemberid"].values
            if other_id in item_to_idx
        ]
        if not target_items:
            users_skipped += 1
            continue

        target = target_items[0]

        ranked_items = _topk_for_user(
            user_idx,
            k=k,
            interaction=interaction,
            user_factors=user_factors,
            item_factors=item_factors,
            extra=500,
        )
        if not ranked_items:
            users_skipped += 1
            continue

        if target in ranked_items:
            hits.append(1.0)
            rr = 1.0 / (ranked_items.index(target) + 1)
        else:
            hits.append(0.0)
            rr = 0.0

        reciprocal_ranks.append(rr)

    users_evaluated = len(grouped) - users_skipped
    hit_rate = np.mean(hits) if hits else np.nan
    mrr = np.mean(reciprocal_ranks) if reciprocal_ranks else np.nan

    return {
        "users_evaluated": users_evaluated,
        "users_skipped": users_skipped,
        f"hit_rate@{k}": float(hit_rate),
        f"mrr@{k}": float(mrr),
    }


metrics = evaluate_model(
    interaction_matrix,
    val_swipes,
    user_to_idx,
    item_to_idx,
    user_factors,
    item_factors,
    k=10,
)
metrics


{'users_evaluated': 39369,
 'users_skipped': 1167,
 'hit_rate@10': 0.03149686301404658,
 'mrr@10': 0.011415939622142471}

## 6. Add a simple profile view and user facing recommender

To make the output easier to read, I build a tiny profile table keyed by
`memberid` with gender and approximate age.

Then I add:

- `describe_user(member_id)` to show a quick summary,
- `recommend_for_member(member_id, k)` to return a small list of suggested
  profiles for that member.


In [11]:
decider_meta = swipes[[
    "decidermemberid",
    "decidergender",
    "deciderdobyear",
]].rename(columns={
    "decidermemberid": "memberid",
    "decidergender": "gender",
    "deciderdobyear": "dobyear",
})

other_meta = swipes[[
    "othermemberid",
    "othergender",
    "otherdobyear",
]].rename(columns={
    "othermemberid": "memberid",
    "othergender": "gender",
    "otherdobyear": "dobyear",
})

meta = pd.concat([decider_meta, other_meta], ignore_index=True)
meta = meta.dropna(subset=["memberid"]).drop_duplicates(subset=["memberid"], keep="first")
user_profiles = meta.set_index("memberid")


In [12]:
def describe_user(member_id: int) -> dict:
    """
    Summarise a user as:
    - memberid
    - gender
    - approximate age (relative to 2021)
    """
    if member_id not in user_profiles.index:
        return {"memberid": member_id, "gender": "unknown", "age": np.nan}

    row = user_profiles.loc[member_id]
    dobyear = row.get("dobyear")
    age = REFERENCE_YEAR - int(dobyear) if pd.notnull(dobyear) else np.nan

    return {
        "memberid": int(member_id),
        "gender": row.get("gender", "unknown"),
        "age": age,
    }


def recommend_for_member(member_id: int, k: int = 5) -> list[dict]:
    """
    Generate top k recommended partner profiles for a given memberid.
    Raises a ValueError if the user has no training history.
    """
    if member_id not in user_to_idx:
        raise ValueError("User not present in the training set.")

    user_idx = user_to_idx[member_id]
    if interaction_matrix[user_idx].nnz == 0:
        raise ValueError("User has no positive training interactions.")

    item_indices = _topk_for_user(
        user_idx,
        k=k,
        interaction=interaction_matrix,
        user_factors=user_factors,
        item_factors=item_factors,
    )
    member_ids = [idx_to_item[i] for i in item_indices]
    return [describe_user(mid) for mid in member_ids]


Now pick a few users who:

- appear in the training set,
- have at least one like,
- get non empty recommendations,

and inspect their top 5 suggestions.


In [13]:
candidate_users = []
for uid, cnt in swipes_like["decidermemberid"].value_counts().items():
    if uid not in user_to_idx:
        continue
    user_idx = user_to_idx[uid]
    if interaction_matrix[user_idx].nnz == 0:
        continue
    if _topk_for_user(
        user_idx,
        k=5,
        interaction=interaction_matrix,
        user_factors=user_factors,
        item_factors=item_factors,
    ):
        candidate_users.append(uid)
    if len(candidate_users) >= 3:
        break

candidate_users


[679634, 147887, 855632]

In [14]:
for member_id in candidate_users:
    print("\n=== User ===")
    print(describe_user(member_id))

    recs = recommend_for_member(member_id, k=5)
    print("Top 5 recommendations:")
    for r in recs:
        print("  ", r)



=== User ===
{'memberid': 679634, 'gender': 'M', 'age': 32}
Top 5 recommendations:
   {'memberid': 1094900, 'gender': 'F', 'age': 24}
   {'memberid': 1415334, 'gender': 'F', 'age': 28}
   {'memberid': 2174890, 'gender': 'F', 'age': 25}

=== User ===
{'memberid': 147887, 'gender': 'M', 'age': 32}
Top 5 recommendations:
   {'memberid': 1671456, 'gender': 'F', 'age': 29}
   {'memberid': 3874078, 'gender': 'F', 'age': 23}
   {'memberid': 3866635, 'gender': 'F', 'age': 27}
   {'memberid': 2946285, 'gender': 'F', 'age': 20}
   {'memberid': 3632714, 'gender': 'F', 'age': 22}

=== User ===
{'memberid': 855632, 'gender': 'M', 'age': 28}
Top 5 recommendations:
   {'memberid': 320277, 'gender': 'F', 'age': 30}


## 7. How this would run in production

A realistic setup around this basic model would look like this.

**Data and training**

- Stream swipe events into a central event log with user IDs, timestamps,
  device, and location context.
- On a schedule (for example nightly), build an implicit interaction matrix
  focused on high quality positive signals such as likes and maybe strong
  engagement features.
- Train or refresh the matrix factorisation model and export user and item
  embeddings into a feature store or a key value store.
- **Model choice**: I chose truncated SVD here for simplicity and speed, but
  alternatives like ALS (Alternating Least Squares), neural two-tower models,
  or user-user collaborative filtering (Jaccard similarity) could also work.
  SVD balances expressiveness with computational efficiency for this scale.

**Serving recommendations**

- When a user opens the app, fetch their embedding or a cold start default.
- Build a candidate pool of profiles based on:
  - gender and orientation preferences,
  - age and location constraints,
  - recent activity,
  - not already swiped on or blocked.
- Score those candidates by dot product with the user embedding.
- Apply business rules and diversity controls, for example:
  - avoid showing near duplicates,
  - avoid only surfacing the most popular people.
- Serve the next set of recommended profiles into the swipe stack.

**Feedback loop and next steps**

- Log which recommended profiles cause likes, matches and conversations.
- Add a re ranking model that learns directly on match or conversation
  rate on top of this latent score.
- Add time decay so older likes count less than recent ones.
- Improve cold start with content or profile features when swipe history
  is sparse.
- At scale, use an ANN index for fast nearest neighbour search on the
  embeddings.


---

## How I used AI tools

I used an AI assistant to help with wording and structure of the notebook.

All modelling choices, evaluation setup and code were reviewed and understood by me.
